# Chapter 11 &mdash; The Formal CFG $(N,\Sigma,S,P)$

**Concept 3 of the Chapter 11 decomposition:** *The Formal CFG $(N,\Sigma,S,P)$: Nonterminals, Terminals, Sentential Forms*

Productions rewrite a nonterminal into a string over $(N\cup\Sigma)^*$; terminal-only forms are sentences.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Formal-CFG/Concept-Formal-CFG.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A context-free grammar is $(N,\Sigma,S,P)$:

* $N$ &mdash; **nonterminals** (syntactic categories),
* $\Sigma$ &mdash; **terminals** (the alphabet of the language), disjoint from $N$,
* $S \in N$ &mdash; the **start symbol**,
* $P$ &mdash; **productions** $A \to \alpha$ with $A\in N$ and
  $\alpha \in (N\cup\Sigma)^*$.

"**Context-free**" names the restriction: the left-hand side is a **single**
nonterminal, so $A$ may be rewritten wherever it appears, regardless of its
surroundings.

A **sentential form** is any string over $(N\cup\Sigma)^*$ reachable from $S$; one
containing only terminals is a **sentence**.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### A grammar, and its four components

In [ ]:
G = mkg({'S': ["aSb", "aB", ""],
         'B': ["b", "bB"]})
show(G)

### Sentential forms, by one step of rewriting

In [ ]:
def step(G, form):
    out = set()
    for i, x in enumerate(form):
        if x in G['N']:
            for r in G['P'][x]:
                out.add(form[:i] + ''.join(r) + form[i+1:])
    return sorted(out)

def forms(G, rounds=3):
    cur, seen = {G['S']}, {G['S']}
    for _ in range(rounds):
        nxt = {f for x in cur for f in step(G, x)} - seen
        seen |= nxt; cur = nxt
    return sorted(seen, key=lambda s: (len(s), s))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch11&nbsp;2.&nbsp;Why a Recursive (or Stack-Based) Mechanism is Needed](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Why-Recursion-Is-Needed/Concept-Why-Recursion-Is-Needed.ipynb) &nbsp;&middot;&nbsp; [**Chapter 11** index](https://github.com/ganeshutah/Jove/blob/master/Chapter11/README.md) &nbsp;&middot;&nbsp; [Ch11&nbsp;4.&nbsp;Derivation Sequences, Parse Trees, and the Language of a CFG](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Derivations-And-Parse-Trees/Concept-Derivations-And-Parse-Trees.ipynb)&nbsp;&rarr;

---

## 3. Tests

The four components are exactly what `mkg` computes.

In [ ]:
print("N     :", sorted(G['N']))
print("Sigma :", sorted(G['Sigma']))
print("S     :", G['S'])
print("|P|   :", sum(len(v) for v in G['P'].values()))
assert G['N'] & G['Sigma'] == set(), "N and Sigma must be disjoint"
assert G['S'] in G['N']

Every production's left side is a **single nonterminal** &mdash; that is 'context-free'.

In [ ]:
for A, rhss in sorted(G['P'].items()):
    assert A in G['N'] and len(A) == 1
    for r in rhss:
        assert all(x in G['N'] or x in G['Sigma'] for x in r)
print("all %d productions have the form A -> alpha with A a single nonterminal"
      % sum(len(v) for v in G['P'].values()))

Sentential forms versus **sentences**.

In [ ]:
F = forms(G, 3)
sent  = [f for f in F if all(x not in G['N'] for x in f)]
mixed = [f for f in F if any(x in G['N'] for x in f)]
print("sentential forms (first 10) :", F[:10])
print("sentences among them        :", sent[:8])
print("still containing a nonterminal :", mixed[:8])
assert set(sent) <= set(language(G, 6) + [''])

The language is the set of sentences.

In [ ]:
L = language(G, 6)
print("L(G) up to length 6 :", L)
assert all(all(x in G['Sigma'] for x in w) for w in L)
print("\nevery member is terminal-only, by definition of L(G)")

Context-freedom in action: the same $B$ rewrites the same way in two places.

In [ ]:
G2 = mkg({'S': ["ABA"], 'A': ["a"], 'B': ["b", "bb"]})
print("L(G2) :", language(G2, 5))
assert set(language(G2, 5)) == {'aba', 'abba'}
print("\nB's options do not depend on the A's around it -- that is the restriction.")

## 4. Exercises


1. Give a grammar where $N$ and $\Sigma$ would overlap if you were careless. What breaks?
2. What would a **context-sensitive** production look like?
3. How many sentential forms of length 4 does `G` have?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter11/Concept-Formal-CFG')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')